In [1]:
# Load the Document
from langchain.document_loaders import PyPDFLoader
loader = PyPDFLoader('the_nestle_hr_policy_pdf_2012.pdf')
document = loader.load()

In [2]:
#Split by Character using CharacterTextSplitter
from langchain.text_splitter import CharacterTextSplitter
textSplitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000) #1000 is the default value

In [3]:
#Create documents with specified chunk size
textPreprocessedData = textSplitter.create_documents([document[0].page_content])

In [4]:
# To check version of pysqlite3
import sqlite3
print(sqlite3.sqlite_version)

3.31.1


In [5]:
import sys
import pysqlite3
sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

In [6]:
#Embedding
from langchain.embeddings import OpenAIEmbeddings
embeddingModel = OpenAIEmbeddings()

In [7]:
#Pass EmbeddingModel and Document to ChromaDB
from langchain.vectorstores import Chroma
vector_store = Chroma.from_documents(textPreprocessedData,embeddingModel,persist_directory='data_db')

In [8]:
#Setup Retriever
retriever = vector_store.as_retriever()

In [11]:
#Build the Question-Answering System
#Enable LLM Chain for QA
from langchain.chains import RetrievalQA
import openai

retriever = vector_store.as_retriever()
template = """
You are an AI assistant for Nestlé's HR department. Use the context below to answer the question accurately.

Context: {context}
Question: {question}
Answer:
"""
def retrieve_and_answer(question):
    # Retrieve relevant chunks
    relevant_docs = retriever.get_relevant_documents(question)
    context = "\n".join([doc.page_content for doc in relevant_docs])
    
    # Use GPT-3.5 Turbo to generate an answer
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": template.format(context=context, question=question)},
        ]
    )
    return response["choices"][0]["message"]["content"].strip()


In [12]:
question = "What is Nestlé's leave policy?"
answer = retrieve_and_answer(question)
print(f"Q: {question}\nA: {answer}")

Q: What is Nestlé's leave policy?
A: Nestlé's leave policy is outlined in the Nestlé Human Resources Policy document which was made mandatory in September 2012. The details of the policy including types of leave, eligibility criteria, and any specific guidelines are likely defined within this document. I recommend referring to the Nestlé Human Resources Policy for specific information on the company's leave policy.


In [13]:
# Create a prompt template to guide the chatbot in understanding and responding to users.
def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message["content"]


In [14]:

prompts = [
    "You are an AI assistant for Nestlé's HR department.",
    "Please tell about Nestlé's HR policies."
]

for prompt in prompts:
    response = get_completion(prompt)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}")

Prompt: You are an AI assistant for Nestlé's HR department.
Response: Hello! How can I assist you today in Nestlé's HR department?
Prompt: Please tell about Nestlé's HR policies.
Response: Nestlé is a multinational food and beverage company that is known for its strong commitment to its employees and their well-being. The company has a comprehensive set of HR policies in place to ensure that its employees are treated fairly and have access to opportunities for growth and development.

Some key aspects of Nestlé's HR policies include:

1. Diversity and Inclusion: Nestlé is committed to creating a diverse and inclusive workplace where all employees feel valued and respected. The company actively promotes diversity and inclusion through its recruitment and retention practices, as well as through training and development programs.

2. Health and Safety: Nestlé places a high priority on the health and safety of its employees. The company has strict policies and procedures in place to ensure

In [17]:
#Step 7: Build the Gradio Interface
interface = gr.Interface(
    fn=get_completion,  # Function to handle queries
    inputs=gr.Textbox(lines=2, placeholder="Ask about Nestlé's HR policies..."),  # Input field
    outputs=gr.Textbox(label="Answer"),  # Output field
    title="Nestlé HR Chatbot",
    description="Ask any question related to Nestlé's HR policies, and the chatbot will provide an answer."
)

# Step 8: Launch the Gradio Interface
interface.launch()

IMPORTANT: You are using gradio version 4.7.1, however version 4.44.1 is available, please upgrade.
--------
Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


In [15]:
#Use Gradio to build a user-friendly chatbot interface, enabling interaction and information retrieval.
import gradio as gr
def chatbot(query):
    response = retrieve_and_answer(query)
    return response["result"]

interface = gr.Interface(
    fn=chatbot,
    inputs=["text", "slider"],
    outputs=["text"],
    # inputs=gr.Textbox(label="Ask a question about Nestlé's HR policies"),
    # outputs=gr.Textbox(label="Answer"),
    title="Nestlé HR Chatbot"
)



IMPORTANT: You are using gradio version 4.7.1, however version 4.44.1 is available, please upgrade.
--------


/usr/local/lib/python3.10/site-packages/gradio/utils.py:828: UserWarning: Expected 1 arguments for function <function chatbot at 0x7f53c233bb50>, received 2.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/gradio/utils.py:836: UserWarning: Expected maximum 1 arguments for function <function chatbot at 0x7f53c233bb50>, received 2.
  warnings.warn(


In [44]:
# Launch the Gradio app
interface.launch(share=True)

Running on local URL:  http://127.0.0.1:7862

Could not create share link. Missing file: /usr/local/lib/python3.10/site-packages/gradio/frpc_linux_amd64_v0.2. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.2/frpc_linux_amd64
2. Rename the downloaded file to: frpc_linux_amd64_v0.2
3. Move the file to this location: /usr/local/lib/python3.10/site-packages/gradio
